In [ ]:
import re
import ast 
import pickle
import json
import os
from omegaconf import OmegaConf

In [ ]:
import json

def normalize_answer(value):
    if value is None:
        return ""
    
    text = str(value).lower().strip()
    

    if text.endswith('%'):
        text = text.replace('%', '')
    

    try:
        number_val = float(text)
        return number_val
    except ValueError:
        return text

def is_nearly_equal(val1, val2, precision=4):
    return val1 == val2


def calculate_accuracy_(file_path):
    total_count = 0
    stats = {
        "first": 0,
        "refine_1": 0,
        "refine_2": 0,
        "refine_3": 0
    }
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                
                try:
                    item = json.loads(line)
                    total_count += 1
                    
                
                    raw_true = item.get("true_answer")
                    norm_true = normalize_answer(raw_true)
                    
        
                    raw_first = item.get("first_result")
                    norm_first = normalize_answer(raw_first)
                    if is_nearly_equal(norm_true, norm_first, precision=2):
                        stats["first"] += 1
                    
                    raw_final_list = item.get("final_result")
                    if not isinstance(raw_final_list, list):
                        raw_final_list = []
                    
                    for i in range(3):
                        refine_key = f"refine_{i+1}"
                        
                        if i < len(raw_final_list) and raw_final_list[i] is not None:
                            current_refine_val = raw_final_list[i]
                        else:
                            current_refine_val = raw_first if i == 0 else last_val
                        
                        norm_refine = normalize_answer(current_refine_val)
                        if is_nearly_equal(norm_true, norm_refine, precision=2):
                            stats[refine_key] += 1
                        
                        last_val = current_refine_val

                except json.JSONDecodeError:
                    print(f"Warning: NO {line_num} row JSON error")
                    
    except FileNotFoundError:
        print(f"Error: file npt exist {file_path}")
        return None

    if total_count == 0:
        print("Error: no data")
        return

    print("=" * 40)
    print(f"samples: {total_count}")
    print("-" * 40)
    
    stages = [
        ("(First Result)", "first"),
        ("(Refine 1)", "refine_1"),
        ("(Refine 2)", "refine_2"),
        ("(Refine 3)", "refine_3"),
    ]
    
    results = {}
    for label, key in stages:
        acc = stats[key] / total_count
        results[key] = acc
        print(f"{label:<20} | True: {stats[key]:>5} | acc: {acc:.10%}")
    print("=" * 40)
    
    return results

def create_path(num , model_name , type , route_type):
    return  f"../result/run{num}_results/tabbench/{model_name}/tabbench-{model_name}-cols_{type}-routing_{route_type}/result.jsonl"


# How to eval

You can choose to directly pass in the results.josn file, or fill in the relevant configuration in the configuration file if you do not change result path in config.
The parameters for create_path are as follows:
* **num**: task_run_ids in config
* **model_name**: model_name in config
* **type**: col_select_type in config
* **route_type**: routing_strategy in config

In [ ]:
path = create_path("1" , "gpt-4o-mini" , "entity"  ,"hybrid")
_ = calculate_accuracy_(path)